In [1]:
!pip install requests


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
%pip install python-dotenv


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install pandas matplotlib seaborn scikit-learn


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [4]:
!pip install tqdm


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


## something

In [5]:
import pandas as pd

In [6]:
# # --- Imports and setup ---
# import sys
# from pathlib import Path
# from dotenv import load_dotenv

# # allow notebook to find scripts folder
# ROOT = Path.cwd().parents[0]
# sys.path.append(str(ROOT))

# # load environment variables
# load_dotenv(ROOT / ".env")

# # --- Run the pipeline ---
# from scripts.pipeline import epc_download_merge_fast as epc

# # Option 1: just run with defaults (downloads all years)
# epc.main()

# # Option 2: limit to specific years
# # epc.YEARS = {2023, 2024}
# # epc.main()

# navigate to project root and import
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]
sys.path.append(str(ROOT))

from scripts.pipeline import epc_download_merge_fast as epc

In [7]:
DTYPES = {
    "LMK_KEY": "string", "BUILDING_REFERENCE_NUMBER": "string",
    "POSTCODE": "string", "POSTTOWN": "string", "COUNTY": "string",
    "LOCAL_AUTHORITY": "string", "LOCAL_AUTHORITY_LABEL": "string",
    "LODGEMENT_DATE": "string",
    "CURRENT_ENERGY_RATING": "string", "POTENTIAL_ENERGY_RATING": "string",
    "PROPERTY_TYPE": "string", "BUILT_FORM": "string",
    "CONSTRUCTION_AGE_BAND": "string", "MAIN_FUEL": "string",
    "MAIN_HEATING_CONTROLS": "string", "MECHANICAL_VENTILATION": "string",
    "MAINS_GAS_FLAG": "string", "TENURE": "string",
    "TRANSACTION_TYPE": "string"
}

In [8]:
df_raw = pd.read_csv("../data/processed/ew_epc_core.csv")
print(df_raw.shape)
df_raw.head()

/var/folders/gs/6gjhnpdn0zd0p99jkjwdr0700000gn/T/ipykernel_26853/1776338431.py:1: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("../data/processed/ew_epc_core.csv")


(28579263, 30)


,LMK_KEY,POSTCODE,BUILDING_REFERENCE_NUMBER,CURRENT_ENERGY_RATING,POTENTIAL_ENERGY_RATING,CURRENT_ENERGY_EFFICIENCY,POTENTIAL_ENERGY_EFFICIENCY,PROPERTY_TYPE,BUILT_FORM,LOCAL_AUTHORITY,...,NUMBER_HABITABLE_ROOMS,NUMBER_HEATED_ROOMS,LOW_ENERGY_LIGHTING,MAIN_FUEL,FLOOR_HEIGHT,MECHANICAL_VENTILATION,LOCAL_AUTHORITY_LABEL,POSTTOWN,CONSTRUCTION_AGE_BAND,TENURE
0,158921618712008100814411800089451,HG5 0HN,4145802568,C,C,78,80,Flat,Mid-Terrace,E07000165,...,3.0,3.0,0.0,mains gas - this is for backwards compatibilit...,2.45,natural,Harrogate,KNARESBOROUGH,England and Wales: before 1900,owner-occupied
1,195706452852008120514220303089255,CV8 2RS,2595235568,C,C,78,80,House,Detached,E07000222,...,8.0,8.0,33.0,mains gas - this is for backwards compatibilit...,2.30,natural,Warwick,KENILWORTH,England and Wales: 2003-2006,owner-occupied
2,179825039352008121218280402989856,TW2 5QJ,8698924568,D,C,65,73,House,Semi-Detached,E09000027,...,2.0,2.0,25.0,mains gas - this is for backwards compatibilit...,2.48,natural,Richmond upon Thames,TWICKENHAM,England and Wales: 1950-1966,owner-occupied
3,161652460962008101409073452598598,DY2 0AQ,9505642568,D,C,56,75,House,Semi-Detached,E08000027,...,3.0,3.0,0.0,mains gas - this is for backwards compatibilit...,2.40,natural,Dudley,DUDLEY,England and Wales: 1930-1949,owner-occupied
4,43459730262008120111543514588538,L33 9TB,3155394568,C,B,77,81,House,Semi-Detached,E08000011,...,6.0,6.0,1.0,mains gas - this is for backwards compatibilit...,2.30,natural,Knowsley,LIVERPOOL,England and Wales: 2003-2006,rental (private)


In [10]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28579263 entries, 0 to 28579262
Data columns (total 30 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   LMK_KEY                        object 
 1   POSTCODE                       object 
 2   BUILDING_REFERENCE_NUMBER      int64  
 3   CURRENT_ENERGY_RATING          object 
 4   POTENTIAL_ENERGY_RATING        object 
 5   CURRENT_ENERGY_EFFICIENCY      int64  
 6   POTENTIAL_ENERGY_EFFICIENCY    int64  
 7   PROPERTY_TYPE                  object 
 8   BUILT_FORM                     object 
 9   LOCAL_AUTHORITY                object 
 10  COUNTY                         object 
 11  LODGEMENT_DATE                 object 
 12  TRANSACTION_TYPE               object 
 13  ENERGY_CONSUMPTION_CURRENT     int64  
 14  ENERGY_CONSUMPTION_POTENTIAL   float64
 15  CO2_EMISSIONS_CURRENT          float64
 16  CO2_EMISS_CURR_PER_FLOOR_AREA  int64  
 17  CO2_EMISSIONS_POTENTIAL        float64
 18  

In [12]:
file_path = "../data/processed/ew_epc_core.csv"

In [14]:
CORE_COLS = [
    # --- IDs & location ---
    "LMK_KEY", "BUILDING_REFERENCE_NUMBER",
    "POSTCODE", "POSTTOWN", "COUNTY",
    "LOCAL_AUTHORITY", "LOCAL_AUTHORITY_LABEL",

    # --- Dates ---
    "LODGEMENT_DATE",

    # --- Ratings ---
    "CURRENT_ENERGY_RATING", "POTENTIAL_ENERGY_RATING",
    "CURRENT_ENERGY_EFFICIENCY", "POTENTIAL_ENERGY_EFFICIENCY",

    # --- Energy & emissions ---
    "ENERGY_CONSUMPTION_CURRENT", "ENERGY_CONSUMPTION_POTENTIAL",
    "CO2_EMISSIONS_CURRENT", "CO2_EMISSIONS_POTENTIAL",
    "CO2_EMISS_CURR_PER_FLOOR_AREA", "TOTAL_FLOOR_AREA",

    # --- Property characteristics ---
    "PROPERTY_TYPE", "BUILT_FORM", "CONSTRUCTION_AGE_BAND",
    "NUMBER_HABITABLE_ROOMS", "NUMBER_HEATED_ROOMS",
    "FLOOR_HEIGHT", "MECHANICAL_VENTILATION",

    # --- Heating & lighting ---
    "MAIN_FUEL", "MAIN_HEATING_CONTROLS", "LOW_ENERGY_LIGHTING",

    # --- Socioeconomic ---
    "TENURE", "TRANSACTION_TYPE"
]

In [16]:
import pandas as pd

chunksize = 500_000
dup_lmk_total = 0
rows_total = 0
missing_totals = None

for chunk in pd.read_csv(file_path, usecols=CORE_COLS, chunksize=chunksize, low_memory=False):
    rows_total += len(chunk)
    dup_lmk_total += chunk.duplicated(subset=["LMK_KEY"]).sum()
    miss = chunk.isna().sum()
    missing_totals = miss if missing_totals is None else missing_totals.add(miss, fill_value=0)

print("Total rows:", rows_total)
print("Duplicate LMK_KEYs:", dup_lmk_total)
print("\nMissing values (%):")
print((missing_totals / rows_total * 100).round(2))

Total rows: 28579263
Duplicate LMK_KEYs: 0

Missing values (%):
LMK_KEY                           0.00
POSTCODE                          0.00
BUILDING_REFERENCE_NUMBER         0.00
CURRENT_ENERGY_RATING             0.00
POTENTIAL_ENERGY_RATING           0.00
CURRENT_ENERGY_EFFICIENCY         0.00
POTENTIAL_ENERGY_EFFICIENCY       0.00
PROPERTY_TYPE                     0.00
BUILT_FORM                        0.02
LOCAL_AUTHORITY                   0.01
COUNTY                           53.56
LODGEMENT_DATE                    0.00
TRANSACTION_TYPE                  0.00
ENERGY_CONSUMPTION_CURRENT        0.00
ENERGY_CONSUMPTION_POTENTIAL      0.03
CO2_EMISSIONS_CURRENT             0.00
CO2_EMISS_CURR_PER_FLOOR_AREA     0.00
CO2_EMISSIONS_POTENTIAL           0.00
TOTAL_FLOOR_AREA                  0.00
MAIN_HEATING_CONTROLS            37.44
NUMBER_HABITABLE_ROOMS           11.93
NUMBER_HEATED_ROOMS              11.93
LOW_ENERGY_LIGHTING               3.21
MAIN_FUEL                         0.60


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots a bit nicer
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# Ensure we can import config.py from project root
ROOT = Path.cwd().parents[0]  # if notebook is in notebooks/
sys.path.append(str(ROOT))

from config import EPC_CLEAN_SW_DIR  # directory with part_*.parquet

EPC_CLEAN_SW_DIR